## <font color=blue> TASK 1: Install and setup LangChain
Welcome to this project notebook, which will serve as your guide to constructing your inaugural Generative AI application. Within this notebook, you'll encounter concise descriptions of each task to enhance your comprehension of the sequence. Our initial task commences with the importation of essential libraries required for this project.

***Note: Before importing the libraries please ensure that all the library modules such as Langchain, Streamlit, PyPdf and other required libraries are installed on this notebook by running the pip command as shown below.***

In [1]:
#For installing the Langchain module associated with OpenAI LLM model
!pip install langchain
!pip install langchain_openai
#For installing the Python library responsible for PDF upload
!pip install pypdf
#For installing the library responsible for web app development
!pip install streamlit
#For installing tokeniser library that asists with converting text strings into tokens recognizable by OpenAI models
!pip install tiktoken
#For installing the library used for invoking the environment file containing secret API key
!pip install python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 120.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 93.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [streamlit]


In [11]:
!pip install -qU langchain-google-genai

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
!pip install -U langchain-community pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 77.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.6 MB/s  0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [langchain-community]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [3]:
import os
import dotenv

from langchain_openai import OpenAI
from langchain_openai.chat_models import ChatOpenAI

from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.chains.summarize import load_summarize_chain

import streamlit as st

/tmp/ipykernel_4456/2024030614.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [12]:
from langchain_google_genai import ChatGoogleGenerativeAI

## <font color=blue> Load OpenAI API Key to access LLM model

## <font color=black>
1. Go to https://platform.openai.com/api-keys
2. Click on the '+ Creat new secret key button'
3. Enter an identifier name(optional) and click on the "Create secret key" button
4. Copy the API key to be used in the API.env file that you need to upload to Google Colab environment

In [4]:
# Load the .env file and invoke the secret API key from the file

# <font color=blue> Load PDF file

In [5]:
from google.colab import files

uploaded = files.upload()

Saving db.pdf to db.pdf


In [6]:
loader = PyPDFLoader("db.pdf")

pages = loader.load()

print(len(pages))
print(pages[0].page_content[:1000])

10
COVID-19 Chest X-Ray Image Classification Using Deep Learning
Gunther Correia Bacellar,1 Mallikarjuna Chandrappa,1 Rajlakshman Kulkarni,1 Soumava Dey1* 
Department of Computer Science, University of Illinois at Urbana-Champaign, Urbana, IL, 61801, USA  
*Correspondence: soumava2@illinois.edu; soumavadey87@gmail.com 
               
ABSTRACT 
The rise of the coronavirus disease 2019 (COVID-19) pandemic has made it necessary to improve existing medical screening 
and clinical management of this disease. While COVID-19 patients are known to exhibit a variety of symptoms, the major 
symptoms include  fever, cough, and fatigue. Since these symptoms also appear in pneumonia patients, this creates 
complications in COVID-19 detection especially during the flu season. Early studies identified abnormalities in chest X -ray 
images of COVID -19 infected patient s that could be beneficial for disease diagnosis. Therefore, chest X -ray image -based 
disease classification has emerged as an alte

## <font color=blue> TASK 2: Define the summarize pdf function
Define the main function that will take pdf file path as an input and generate a summary of the file.

In [17]:
from langchain_core.runnables import chain
def summarize_pdf(pdf_file_path, chunk_size, chunk_overlap):

    #Instantiate LLM model
    llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    temperature=0,
    google_api_key=""
)
    #Load PDF file
    loader = PyPDFLoader(pdf_file_path)
    docs_raw = loader.load()

    #Create multiple documents
    docs_raw_text = [doc.page_content for doc in docs_raw]

    #Split text into chunks
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    docs_chunks = text_splitter.create_documents(docs_raw_text)

    #Summarize the chunks (map_reduce method takes longer to execute)
    chain = load_summarize_chain(llm, chain_type="stuff")
    #Return the summary
    summary = chain.invoke(docs_chunks, return_only_outputs=True)
    return summary['output_text']




In [18]:
#print summary by using chain type stuff or map_reduce
#Chunk size and chunk overlap values set to random value
pdf_path = "/content/db.pdf"
print(summarize_pdf(pdf_path, 1000, 20))

**Summary:**

To address the limitations of RT-PCR testing and the challenges of manually diagnosing COVID-19 from radiography, researchers developed **DLH_COVID**, a custom convolutional neural network (CNN) designed to classify chest X-ray images into three categories: COVID-19, pneumonia, and normal/healthy. 

**Key Highlights:**
* **Model & Methodology:** The custom architecture—consisting of three convolutional layers and two fully connected layers—was trained on a dataset of 6,432 chest X-ray images. It was optimized using Stochastic Gradient Descent (SGD) and dynamic learning rate scheduling, then benchmarked against several pre-trained models (including ResNet, VGG, Inception, and EfficientNet).
* **Performance:** DLH_COVID outperformed the transfer learning models, achieving roughly 96% accuracy on the validation set and a 95% average accuracy across 5-fold cross-validation. 
* **Deployment:** The model was integrated into a public, cloud-based web application that provides re

## <font color=blue> TASK 3: Add Prompt template to the summarizer function
Leveraging prompt templates to extract key information from the reserach paper in more guided manner.

## <font color=blue> Define Prompt Templates

### <font color=black> Prompt Template for Stuffing chain type

In [19]:
map_prompt_template = """
                       Write a summary of the research paper for an
                       artficial intelligence researcher that includes
                       main points and any important details in bullet points.{text}
                      """

map_prompt = PromptTemplate(template=map_prompt_template, input_variables=["text"])

### <font color=black> Add Combo Template for Map_Reduce chain type

In [ ]:
combine_prompt_template = """
                              You will be given main points and any impotant details of research paper in bullet points.Your goal is to givr msin research topic and finding which will be useful to AI researcher to grasp what was done during research work.
'''{test}'''
FINAL SUMMARY:
"""

combine_prompt =

In [21]:
#Modify the custom function to add the prompt templates

def summarize_pdf(pdf_file_path, chunk_size, chunk_overlap, map_prompt):

    #Instantiate LLM model
    llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    temperature=0,
    google_api_key=""
)

    #Load PDF file
    loader = PyPDFLoader(pdf_file_path)
    docs_raw = loader.load()

    #Create multiple documents
    docs_raw_text = [doc.page_content for doc in docs_raw]

    #Split text into chunks
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    docs_chunks = text_splitter.create_documents(docs_raw_text)

    #Summarize the chunks
    chain = load_summarize_chain(llm, chain_type="stuff",prompt=map_prompt)


    #Return the summary
    summary = chain.invoke(docs_chunks, return_only_outputs=True)
    return summary['output_text']

In [26]:
#print summary using the map_prompt and combine prompt
#Increasing the chunk size value might reduce the overall summarization time with map_reduce method
print(summarize_pdf(pdf_path, 1000,20,map_prompt))

## <font color=blue> TASK 4: Build and test a GenAI app for PDF summarization

In [49]:
!pip install -U langchain-core langchain-text-splitters langchain-classic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 577.1/577.1 kB 21.6 MB/s  0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.7.0
    Uninstalling langchain-core-1.7.0:
      Successfully uninstalled langchain-core-1.7.0


In [70]:
%%writefile app.py
import os
import dotenv
from langchain_openai import OpenAI
#from langchain_openai.chat_models import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.chains.summarize import load_summarize_chain
import streamlit as st


# Load the .env file and invoke the secret API key from the file

#summarize_pdf function

# def summarize_pdf(pdf_file_path, chunk_size, chunk_overlap, prompt):
#     #Instantiate LLM model gpt-3.5-turbo-16k
#      #Instantiate LLM model
llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    temperature=0,
    google_api_key=""
)
import requests
import tempfile
from langchain_community.document_loaders import PyPDFLoader

def summarize_pdf(pdf_url, chunk_size, chunk_overlap, prompt):

    # Download PDF from the URL
    response = requests.get(
        pdf_url,
        headers={
            "User-Agent": "Mozilla/5.0",
            "Accept": "application/pdf,*/*"
        },
        timeout=60
    )
    response.raise_for_status()

    # Verify that the response is actually a PDF
    if not response.content.startswith(b"%PDF"):
        raise ValueError("The link did not return a valid PDF.")

    # Save PDF temporarily
    with tempfile.NamedTemporaryFile(
        suffix=".pdf", delete=False
    ) as temp:
        temp.write(response.content)
        pdf_path = temp.name

    # Load the downloaded PDF
    loader = PyPDFLoader(pdf_path)
    docs_raw = loader.load()

    # Split into chunks
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    docs_chunks = text_splitter.split_documents(docs_raw)

    # Gemini summarization
    chain = load_summarize_chain(
        llm,
        chain_type="stuff",
        prompt=prompt
    )

    summary = chain.invoke(
        {"input_documents": docs_chunks}
    )

    return summary["output_text"]

    # #Load PDF file
    # loader = PyPDFLoader(pdf_file_path)
    # docs_raw = loader.load()

    # #Create multiple documents
    # docs_raw_text = [doc.page_content for doc in docs_raw]

    # #Split text into chunks
    # text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    # docs_chunks = text_splitter.create_documents(docs_raw_text)

    # #Summarize the chunks
    # chain = load_summarize_chain(llm, chain_type="stuff", prompt = prompt)
    # #Return the summary
    # summary = chain.invoke(docs_chunks, return_only_outputs=True)
    # return summary['output_text']

#streamlit app main() function

def main():
    #Set page config and title
    st.set_page_config(page_title="PDF Summarizer", page_icon=":book:", layout="wide")
    st.title("PDF Summarizer ")

  #Add custom sliders and selectbox for more user interaction


  #Display warning message

    #Input pdf file path
    pdf_file_path = st.text_input("Enter PDF file path:")
    if pdf_file_path is "":
      st.write("Pdf loaded successfully")
    #prompt input
    user_prompt = st.text_input("Enter prompt:")
    user_prompt=user_prompt+"""{text}"""
    prompt = PromptTemplate(template=user_prompt, input_variables=["text"])
    #Summarize button
    if st.button("Summarize"):
        #Summarize pdf
        summary = summarize_pdf(pdf_file_path, 1000, 20, prompt)
        st.write(summary)
if __name__ == "__main__":
    main()

Overwriting app.py


## <font color=blue> Launch Streamlit app from Google Colab

The following lines of code would enable users to launch Streamlit app from Google Colab using [ngrok service](https://ngrok.com/)

In [28]:
!wget https://bin.equinox.io/c/4VmDzA7iaHb/ngrok-stable-linux-amd64.zip

--2026-11-01 04:42:01--  https://bin.equinox.io/c/4VmDzA7iaHb/ngrok-stable-linux-amd64.zip
Resolving bin.equinox.io (bin.equinox.io)... 13.248.244.96, 35.71.179.82, 99.83.220.108, ...
Connecting to bin.equinox.io (bin.equinox.io)|13.248.244.96|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 13921656 (13M) [application/octet-stream]
Saving to: ‘ngrok-stable-linux-amd64.zip’

ngrok-stable-linux- 100%[===================>]  13.28M  6.16MB/s    in 2.2s    

2026-11-01 04:42:04 (6.16 MB/s) - ‘ngrok-stable-linux-amd64.zip’ saved [13921656/13921656]



In [29]:
!unzip ngrok-stable-linux-amd64.zip

Archive:  ngrok-stable-linux-amd64.zip
  inflating: ngrok                   


In [30]:
get_ipython().system_raw('./ngrok http 8501 &')

In [75]:
!wget -q -O - ipv4.icanhazip.com

In [32]:
# !streamlit run app.py & npx localtunnel --port 8501



⠙⠹⠸⠼⠴Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) 2026-11-01 04:42:34.397 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.247.148.243:8501

  Stopping...
^C


In [33]:
# !streamlit run app.py --server.port 8501 &



2026-11-01 04:44:02.202 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.247.148.243:8501

  Stopping...


In [34]:
# !npm install -g localtunnel
# !lt --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼
added 22 packages in 1s
⠼
⠼3 packages are looking for funding
⠼  run `npm fund` for details
⠼your url is: https://sharp-horses-take.loca.lt
^C


In [55]:
!nohup streamlit run app.py --server.address 0.0.0.0 --server.port 8501 > /tmp/streamlit.log 2>&1 &

In [56]:
# !curl -I http://127.0.0.1:8501

HTTP/1.1 200 OK
date: Sun, 01 Nov 2026 05:07:45 GMT
server: uvicorn
content-type: text/html; charset=utf-8
accept-ranges: bytes
content-length: 6467
last-modified: Sun, 01 Nov 2026 03:55:15 GMT
etag: "2df4b86e3239b54167dd4e9e6c0c7402"
cache-control: no-cache



In [57]:
# !npx localtunnel --port 8501

⠙⠹⠸⠼your url is: https://rotten-rings-remain.loca.lt
^C


In [58]:
!pip install -q pyngrok

In [74]:
from pyngrok import ngrok

ngrok.set_auth_token("")
public_url = ngrok.connect(8501)
print(public_url)

In [71]:
!nohup streamlit run app.py --server.address 0.0.0.0 --server.port 8501 > /tmp/streamlit.log 2>&1 &

In [72]:
!pkill -f streamlit

In [73]:
!nohup streamlit run app.py --server.address 0.0.0.0 --server.port 8501 > /tmp/streamlit.log 2>&1 &

Click the link to explore useful Streamlit library functions:
https://docs.streamlit.io/library/cheatsheet